# <b>Superlet Burst Detection Pipeline</b>

### ORIGINAL ALGORITHM & SCRIPT:
- Authors: danclab (Maciej Szul et al.)
- Original Repository: https://github.com/danclab/burst_detection

### MODIFICATIONS BY: Juliette Leclerc

### DESCRIPTION OF CHANGES & UPGRADES:
#### This script significantly expands upon the original burst detection example to create a robust, automated pipeline for complex in vivo electrophysiology data. Major modifications include:

1. **Batch Processing & file selection GUI:** Replaced hardcoded single-file paths with an interactive dialog (tkinter) for dynamic batch folder selection.
2. **Complex Data Handling:** Integrated `neo.io` and `mne` to directly load multi-channel .mat files, preserve 2D spatial coordinates, and automatically drop bad epochs (NaNs).
3. **Dynamic Aperiodic (1/f) Subtraction:** Implemented a sliding-window FOOOF (specparam) fit across trials to dynamically calculate and subtract the aperiodic background from the time-frequency data before burst extraction.
4. **Automated Export:** Added custom functions to flatten complex burst dictionaries (including 2D waveforms and scalar features) into structured pandas DataFrames for automated .csv and .xlsx export.
5. **Enhanced Visualization:** Upgraded plotting suite to include multi-epoch looping, direct comparisons of raw vs. FOOOF-corrected PSDs, and high resolution time-frequency maps with overlaid burst peaks.
6. **Modular Architecture:** Encapsulated the entire workflow into a master `pipeline()` function controlled by a centralized `config` dictionary for rapid parameter adjustments.

#### <b>Warning</b> : The files superlet.py and burst_detection_modified.py must be located in the same folder as this Jupyter notebook.

In [ ]:
from matplotlib import pyplot as plt
from Superlet import superlet, scale_from_period
import mne
import numpy as np
from burst_detection import extract_bursts
from specparam import SpectralModel

#Neo
import neo
from neo.io import NixIO

#tkinter
import tkinter as tk
from tkinter import filedialog

import os
import json
import glob

import quantities as pq
import pickle
from time import time

import pandas as pd
from tqdm import tqdm

## I. Constant Data

In [ ]:
# X coordinates of each channel (in mm )
x = [0.    , 0.0225, 0.    , 0.0225, 0.    , 0.0225, 0.    , 0.0225,
        0.    , 0.0225, 0.    , 0.0225, 0.    , 0.0225, 0.    , 0.0225,
        0.    , 0.0225, 0.    , 0.0225, 0.    , 0.0225, 0.    , 0.0225,
        0.    , 0.0225, 0.    , 0.0225, 0.    , 0.0225, 0.    , 0.0225]

# Y coordinates of each channel (in mm)
y = [0.   , 0.025, 0.025, 0.05 , 0.05 , 0.075, 0.075, 0.1  , 0.1  ,
        0.125, 0.125, 0.15 , 0.15 , 0.175, 0.175, 0.2  , 0.2  , 0.225,
        0.225, 0.25 , 0.25 , 0.275, 0.275, 0.3  , 0.3  , 0.325, 0.325,
        0.35 , 0.35 , 0.375, 0.375, 0.4  ]

# List of channel names, ordered to match x and y positions
ch_names = [
    'CH7', 'CH10', 'CH6', 'CH5', 'CH4', 'CH3', 'CH15', 'CH16',
    'CH1', 'CH2', 'CH14', 'CH13', 'CH18', 'CH12', 'CH17', 'CH11',
    'CH32', 'CH9', 'CH31', 'CH8', 'CH19', 'CH26', 'CH20', 'CH23',
    'CH21', 'CH27', 'CH22', 'CH28', 'CH24', 'CH29', 'CH25', 'CH30'
]

## II. Loading data

### 1) File choice

In [ ]:
def file_choice():
    """ Opens a folder dialog repeatedly to let the user choose folders.
        Returns a list of all '_NaN_neo.mat' files from all selected folders.
    
    Output:
        - files : list of str, full paths to the selected files
    """

    all_files = []

    while True:
        # Create a hidden Tkinter root window (required for dialogs)
        root = tk.Tk()
        root.attributes("-topmost", True)
        root.update()
        root.withdraw()

        # Ask the user to select a folder
        folder_path = filedialog.askdirectory(title="Indicate where to find neo files (Cancel to finish)")

        # Destroy the hidden root window
        root.destroy()

        # If user cancels, break the loop
        if not folder_path:
            break

        # Add all matching files from this folder
        files = glob.glob(f"{folder_path}/*_neo_NaN.mat")
        all_files.extend(files)

    # Print the number of files detected
    print(f"{len(all_files)} file{'s' if len(all_files) != 1 else ''} detected")

    return all_files


### 2) Create Neo File and MNE files

In [ ]:
def create_neo_file(x, y, ch_names,epoch_duration,file_path):
    """ Loads a Neo .mat file and its associated metadata, annotates the signal, and converts it into an MNE Raw object 
    with channel-level information.
    Inputs :
        - x : list or array of x-coordinates for each channel
        - y : list or array of y-coordinates for each channel
        - ch_names : list of str, names of the channels
    Outputs :
        - raw : MNE RawArray object containing the signal and metadata
        - info : MNE Info object containing metadata about the recording
        - regions_dict : dict, mapping of regions to channels based on distance to lesion
        - file_path : str, path to the original .mat file
    """

    # Prompt the user to select the Neo .mat file
    data_path = os.path.dirname(file_path)
    print("Selected file:", file_path)

    # Load the Neo block from the selected .mat file
    r = neo.io.NeoMatlabIO(file_path)
    bl = r.read_block()

    # Get the base filename and extract the signal
    filename = bl.name
    signal = bl.segments[0].analogsignals[0]

    # Construct paths to metadata files: distance to lesion and good/bad channel labels
    good_bad_path = os.path.join(data_path, f"{filename}_bad_good_channels_neo.npy")

    if not os.path.exists(good_bad_path):
        raise FileNotFoundError(f"Missing file: {good_bad_path}")

    # Load metadata
    good_bad_bin_channels = np.load(good_bad_path).flatten()

    # Convert binary labels to string labels: 1 = 'good', 0 = 'bad'
    good_bad_channels = np.where(good_bad_bin_channels == 1, 'good', 'bad')

    # Check consistency between signal and annotations
    n_channels = signal.shape[1]
    if not ( len(good_bad_channels) == n_channels):
        raise ValueError("Mismatch between annotation lengths and number of signal channels.")

    # Annotate the Neo signal with spatial and quality metadata
    signal.array_annotate(good_bad_channels=good_bad_channels)
    signal.annotate(spatial_scale=0.025 * pq.mm)
    signal.array_annotate(x_coords=x, y_coords=y)

    # Convert the Neo signal to a NumPy array in volts and transpose to shape (n_channels, n_times)
    data = signal.rescale("V").magnitude.T
    sfreq = signal.sampling_rate.magnitude

    samples_per_epoch = int(epoch_duration * sfreq)
    n_epochs = data.shape[1] // samples_per_epoch

    events = np.column_stack([
        np.arange(n_epochs) * samples_per_epoch,  # sample onsets
        np.zeros(n_epochs, dtype=int),            # placeholder
        np.ones(n_epochs, dtype=int)              # event ID
    ])

    # Create the MNE Info object with electrode metadata
    ch_types = ["seeg"] * len(ch_names)
    info = mne.create_info(ch_names=ch_names, ch_types=ch_types, sfreq=sfreq)

    # Create the MNE RawArray object from the signal and info
    raw = mne.io.RawArray(data, info)


    # Return all relevant outputs
    return raw,events,sfreq, info

## III. Export data

### 1. transform Bursts data into dataframe for csv exportation

In [ ]:
def to_dataframe(data_dict):
    """
    Converts a burst analysis dictionary into a single pandas DataFrame by combining
    scalar features and waveform data.

    Parameters:
        data_dict (dict): Dictionary containing burst features.
            Expected to include:
                - 1D arrays (e.g. 'peak_freq', 'peak_amp_base', etc.) of shape (n_bursts,)
                - 2D array 'waveform' of shape (n_bursts, n_timepoints)

    Returns:
        pd.DataFrame: A DataFrame where each row corresponds to a burst,
                      including scalar features and one column per waveform timepoint
                      (e.g., wave_0, wave_1, ..., wave_N).
    """

    #Extract all 1D arrays (scalars) that have the same number of rows as the waveform
    ref_len = data_dict['waveform'].shape[0]  # Number of bursts (rows)
    df_scalars = {}
    for key, value in data_dict.items():
        value = np.array(value)
        # Keep only 1D arrays with length equal to number of bursts
        if value.ndim == 1 and len(value) == ref_len:
            df_scalars[key] = value
    df_scalars = pd.DataFrame(df_scalars)  # Convert scalar features to a DataFrame

    # Convert the waveform (2D array) into a DataFrame
    # Each column represents a time point within the burst waveform
    waveform = np.array(data_dict['waveform'])
    waveform_df = pd.DataFrame(
        waveform, columns=[f"wave_{i}" for i in range(waveform.shape[1])]
    )

    # Concatenate scalar features and waveform features into a single DataFrame
    return pd.concat([df_scalars, waveform_df], axis=1)


## IV. Superlet and FOOOF Analysis

In [ ]:
def superlet_analysis(config, done_superlet, file_path, foi, trials, sfreq,ch_to_analyse, max_order=40, n_cycle=4):
    """
    Runs the Superlet transform and FOOOF fitting on trial data for a specific channel.
    Can save or load precomputed results. Produces and optionally visualizes PSD estimates.

    Parameters:
        - config (dict): Configuration dictionary containing channel and band info
        - done_superlet : bool, if True, loads previously saved Superlet output; if False, computes and saves it
        - file_path : str, path to the original data file (used to construct output filenames)
        - foi : 1D array, frequencies of interest used for Superlet and FOOOF
        - trials : array of shape (n_trials, n_times), signal from one selected channel
        - sfreq : float, sampling frequency
        - search_range : array-like of indices into `foi` used for burst detection
        - ch_to_analyse : str, name of the channel being analyzed
        - freq_to_analyse : str, label of the frequency band (e.g., "beta")
        - max_order : int, maximum Superlet order
        - n_cycle : int, number of base cycles used for Morlet wavelets
        - visualisation : bool, whether to generate and save plots

    Returns:
        - tf_trials : array (n_trials, n_freqs, n_times), time-frequency amplitude
        - foi : array, frequencies used
        - ap : array (n_trials,n_freqs) aperiodic (background) PSD fit from FOOOF
        - residual_search_power : array, difference between PSD and aperiodic fit in search band
    """

    if done_superlet == False:
        # Compute Superlet transform for each trial
        scales = scale_from_period(1 / foi)
        tf_trials = []
    
        for trial in tqdm(trials, desc="Superlet Transform"):
            tf = superlet(trial, sfreq, scales, max_order, c_1=n_cycle, adaptive=True)
            tf_trials.append(np.single(np.abs(tf)))  # Take magnitude and convert to float32
        tf_trials = np.array(tf_trials)

        # Save Superlet result to file for future reuse
        np.savez(f"{file_path[:-8]}_{ch_to_analyse}_superlet_data.npz", tf_trials=tf_trials, foi=foi)
        print("##### File saved #####")

    else:
        data = np.load(f"{file_path[:-8]}_{ch_to_analyse}_superlet_data.npz")
        tf_trials = data["tf_trials"]
        foi = data["foi"]
        print("##### File loaded #####")


    # === Average power spectral density over time and trials ===
    average_psd = np.average(tf_trials, axis=(2, 0))  # (n_freqs,)
    
    # === Prepare for FOOOF fitting ===
    n_trials = tf_trials.shape[0]
    n_freqs = tf_trials.shape[1]
    freq_range = [1, 100]

    ap = np.zeros((n_trials, n_freqs))
    offset = np.zeros(n_trials)
    knee = np.zeros(n_trials)
    exponent = np.zeros(n_trials)

    print("##### FOOOF Processing with ±2 trial averaging #####")

    half_window = 2  # number of neighboring trials before/after to include to smooth the fooof spectrum

    for t in range(n_trials):
        # --- define window boundaries (shrink at edges) ---
        start_idx = max(0, t - half_window)
        end_idx = min(n_trials, t + half_window + 1)

        # --- average PSD across trials and time ---
        window_psd = np.mean(tf_trials[start_idx:end_idx, :, :], axis=(0, 2))  # (n_freqs,)

        # --- fit FOOOF model ---
        fm = SpectralModel(aperiodic_mode="knee")
        fm.fit(foi, window_psd, freq_range)

        # --- extract and store parameters ---
        offset_t, knee_t, exponent_t = fm.aperiodic_params_
        ap_t = 10 ** fm._ap_fit  # convert from log to linear

        offset[t] = offset_t
        knee[t] = knee_t
        exponent[t] = exponent_t
        ap[t, :] = ap_t

    # Compute residual PSD in the search range
    #residual_search_power = average_psd[search_range] - ap[search_range]

    print("##### Done: Aperiodic parameters computed with sliding window #####")


    
    # === EXPORT TO EXCEL ===
    # Load plotting parameters from config
    ch_to_analyse = config["ch_to_analyse"]
    freq_to_analyse = config["fta"]
    trial = config["epochs_to_visualize"]
    
    fta_label = config["fta"]
    
    # Generate base file name depending on presence of a label
    file_name=f"{file_path[:-8]}_{ch_to_analyse}_{freq_to_analyse}"

    # Build output filename
    data_path = os.path.dirname(file_path)
    save_path = os.path.join(data_path, f"{file_name}_aperiodic_components.xlsx")

    trial_numbers = np.arange(0, n_trials) * config["epoch_step"]

    # Compute trial (epoch) numbers based on config["epoch_step"]
    trial_numbers = np.arange(0, n_trials) * config["epoch_step"]
    trial_labels = [f"Epoch_{int(num)}" for num in trial_numbers]

    # --- Sheet 1: Aperiodic fits (ap) ---
    # Rows = frequencies (foi), Columns = trials
    df_ap = pd.DataFrame(ap.T, index=foi, columns=trial_labels)
    df_ap.index.name = "Frequency (Hz)"

    # --- Sheet 2: Aperiodic parameters (offset, knee, exponent) ---
    df_params = pd.DataFrame({
        "Offset": offset,
        "Knee": knee,
        "Exponent": exponent
    }).T  # transpose so rows = parameters, columns = trials
    df_params.columns = trial_labels
    df_params.index.name = "Parameter"

    # Write both sheets to Excel
    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        df_ap.to_excel(writer, sheet_name="Aperiodic Fit (ap)")
        df_params.to_excel(writer, sheet_name="Aperiodic Parameters")

    print(f"##### Aperiodic components exported to: {save_path} #####")
    

    return tf_trials, foi, ap,average_psd, offset, knee, exponent


## V. Burst Extraction

### 1. Burst extraction

In [ ]:
def burst_extraction(done_burst, trials, tf_trials, search_range, times, foi, freq_lims,
                     ap, sfreq, file_path, ch_to_analyse, freq_to_analyse, save_csv=True):
    """
    Performs burst extraction on time-frequency transformed data, either by computing
    new bursts or loading them from a previously saved file. Optionally saves results
    as a .pkl and .csv file.

    Parameters:
        done_burst (bool): If False, compute and save bursts; if True, load from file.
        trials (ndarray): Epoched signal data of shape (n_trials, n_timepoints).
        tf_trials (ndarray): Time-frequency representation of shape (n_trials, n_freqs, n_timepoints).
        search_range (array): Indices of frequencies in `foi` to use for burst detection.
        times (array): Time vector corresponding to each timepoint in trials.
        foi (array): Full frequency vector corresponding to tf_trials.
        freq_lims (tuple): Frequency band within which to keep detected bursts (e.g., (8, 12) for alpha).
        ap (array): Aperiodic (background) spectrum to subtract from the PSD.
        sfreq (float): Sampling frequency of the signal.
        file_path (str): Path to the original data file (used to construct filenames).
        ch_to_analyse (str): Name of the analyzed channel (used for file naming).
        freq_to_analyse (str): Name of the frequency band (e.g. 'beta') used for file naming.
        save_csv (bool): If True, save extracted bursts as a CSV file.

    Returns:
        bursts (dict): Dictionary containing burst features for all trials.
    """

    if done_burst == False:
        print("##### Burst Extraction  #####")

        # Perform burst detection using selected frequency range
        bursts = extract_bursts(
            trials,                          # raw signal data
            tf_trials[:, search_range],     # time-frequency data restricted to search_range
            times,                           # time vector
            foi[search_range],               # corresponding frequencies for tf data
            freq_lims,                       # frequency limits for accepted bursts
            ap[:, search_range].reshape(ap.shape[0], -1, 1), # background spectrum reshaped to match tf
            sfreq                            # sampling frequency
        )

        # Save bursts dictionary to file
        with open(f"{file_path[:-8]}_{ch_to_analyse}_{freq_to_analyse}_bursts_data.pkl", "wb") as f:
            pickle.dump(bursts, f)
        print("##### Burst Extraction file saved #####")

    else:
        # Load precomputed bursts from file
        with open(f"{file_path[:-8]}_{ch_to_analyse}_{freq_to_analyse}_bursts_data.pkl", "rb") as f:
            bursts = pickle.load(f)

    # Convert bursts dictionary to DataFrame for easy export
    df_bursts = to_dataframe(bursts)

    # Optionally save the DataFrame to CSV
    if save_csv:
        df_bursts.to_csv(f"{file_path[:-8]}_{ch_to_analyse}_{freq_to_analyse}_bursts_data.csv", index=False)

    return bursts


### 2. Burst extraction for a specific band of frequencies

In [ ]:
def filtered_burst_to_foi(bursts, fmin, fmax):
    """
    Filters a burst dictionary to retain only bursts whose peak frequency falls within a specific frequency band.

    Parameters:
        bursts (dict): Dictionary of burst features, typically the output of extract_bursts().
        fmin (float): Lower bound of the frequency band of interest (Hz).
        fmax (float): Upper bound of the frequency band of interest (Hz).

    Returns:
        filtered_data (dict): Dictionary with the same structure as `bursts`,
                              but only containing bursts within [fmin, fmax] in 'peak_freq'.
                              Non-filterable entries (e.g., waveform_times) are preserved as-is.
    """

    # Create a boolean mask for bursts whose peak frequency is within the desired frequency band
    mask = (bursts['peak_freq'] >= fmin) & (bursts['peak_freq'] <= fmax)

    filtered_data = {}

    for key, value in bursts.items():
        value = np.array(value)

        # Apply the mask only to entries where the first dimension matches the number of bursts
        if value.shape[0] == mask.shape[0]:
            filtered_data[key] = value[mask]
        else:
            # Keep other entries untouched (e.g., waveform_times)
            filtered_data[key] = value

    return filtered_data


## VI. Plot

### 1. PSD Plot

In [ ]:
def plot_psd(foi, average_psd, ap, file_path, config, search_range, save_fig=True):
    """
    Plots the power spectral density (PSD) and the aperiodic fit (from FOOOF),
    both for the full frequency range and zoomed in on the search range.

    Parameters:
        foi (array): Frequencies of interest (used as x-axis for PSD).
        average_psd (array): Average power spectrum across trials.
        ap (array): Aperiodic (background) fit from FOOOF.
        file_path (str): Path to the original file (used for naming output figures).
        config (dict): Configuration dictionary containing channel and band info.
        search_range (array): Indices into `foi` corresponding to the frequency band of interest.
        save_fig (bool): If True, save the plots as PNG files.
    """
    print("##### PSD Visualisation #####")

    # Get relevant info from config
    ch_to_analyse = config["ch_to_analyse"]
    freq_to_analyse = config["fta"]
    foi_label = config["foi"]
    
    # Generate base file name depending on presence of a label
    if foi_label != "":
        file_name = f"{file_path[:-8]}_{ch_to_analyse}_{foi_label}"
    else:
        file_name = f"{file_path[:-8]}_{ch_to_analyse}"

    # --- 1. Plot full PSD with aperiodic (FOOOF) fit ---
    average_ap = np.average(ap,axis=0)

    f, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(foi, average_psd, c="red", label="Averaged PSD")
    ax.plot(foi, average_ap, c="blue", label="FOOOF Aperiodic Fit")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power")
    ax.set_title("Power Spectrum with Aperiodic Fit")
    ax.legend()
    if save_fig:
        f.savefig(f"{file_path[:-8]}_{ch_to_analyse}_average_psd.png", dpi=300, bbox_inches='tight')

    # --- 2. Plot zoomed-in PSD in the selected search frequency band ---
    f, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(foi[search_range], average_psd[search_range], c="red", label="Averaged PSD")
    ax.plot(foi[search_range], average_ap[search_range], c="blue", label="FOOOF Aperiodic Fit")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Power")
    ax.set_title("PSD in Search Frequency Band")
    ax.legend()
    if save_fig:
        f.savefig(f"{file_name}_{freq_to_analyse}_Average_psd_search_range.png", dpi=300, bbox_inches='tight')


### 2. Bursts Plot

In [ ]:
def plot_bursts(bursts, file_path, config, tf_trials, search_range, times, foi, freqs_lims, ap, save_fig=True):
    """
    Generates multiple visualizations related to burst characteristics:
    - Histograms of duration, frequency span, peak frequency, and peak amplitude
    - Overlay of all burst waveforms with average
    - Time-frequency map for a selected epoch with detected burst peaks overlaid

    Parameters:
        bursts (dict): Dictionary containing burst features (output of extract_bursts).
        file_path (str): Path to the original data file (used for naming figures).
        config (dict): Configuration dictionary with plotting parameters.
        tf_trials (ndarray): Time-frequency data of shape (n_trials, n_freqs, n_times).
        search_range (array): Indices into `foi` corresponding to the frequency band of interest.
        times (array): Time vector.
        foi (array): Frequencies of interest.
        freqs_lims (tuple): Frequency band boundaries for horizontal lines on the TF map.
        save_fig (bool): Whether to save the generated plots to file.
    """

    # Load plotting parameters from config
    ch_to_analyse = config["ch_to_analyse"]
    freq_to_analyse = config["fta"]
    epochs_to_visualize = config["epochs_to_visualize"]
    foi=config["foi"]
    if foi !="":
        file_name=f"{file_path[:-8]}_{ch_to_analyse}_{freq_to_analyse}_{foi}"
    else:
        file_name=f"{file_path[:-8]}_{ch_to_analyse}_{freq_to_analyse}"

    print("##### Burst Extraction Visualisation #####")

    # --- Plot 1: Histogram of burst durations ---
    f, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.hist(bursts["fwhm_time"], bins=20, color="#DFFF00", edgecolor='black', linewidth=0.2)
    ax.set_title("Burst Duration")
    if save_fig:
        f.savefig(f"{file_name}_Burst_Duration.png", dpi=300, bbox_inches='tight')

    # --- Plot 2: Histogram of frequency spans ---
    f, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.hist(bursts["fwhm_freq"], bins=10, color="#DFFF00", edgecolor='black', linewidth=0.2)
    ax.set_title("Frequency Span")
    if save_fig:
        f.savefig(f"{file_name}_Frequency_Span.png", dpi=300, bbox_inches='tight')

    # --- Plot 3: Histogram of peak frequencies ---
    f, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.hist(bursts["peak_freq"], bins=18, color="#DFFF00", edgecolor='black', linewidth=0.2)
    ax.set_title("Peak Frequency")
    ax.legend()
    if save_fig:
        f.savefig(f"{file_name}_Peak_Frequency.png", dpi=300, bbox_inches='tight')

    # --- Plot 4: Histogram of peak amplitudes ---
    f, ax = plt.subplots(1, 1, figsize=(7, 5))
    ax.hist(bursts["peak_amp_base"], bins=20, color="#DFFF00", edgecolor='black', linewidth=0.2)
    ax.set_title("Peak Amplitude")
    if save_fig:
        f.savefig(f"{file_name}_Peak_Amplitude.png", dpi=300, bbox_inches='tight')

    # --- Plot 5: Waveforms (all + average) ---
    f, ax = plt.subplots(1, 1, figsize=(5, 5))
    mean_waveform = np.mean(bursts["waveform"], axis=0)
    ax.plot(bursts["waveform_times"], bursts["waveform"].T, lw=0.2)
    ax.plot(bursts["waveform_times"], mean_waveform, lw=1, c="black")
    ax.set_xlim(bursts["waveform_times"][0], bursts["waveform_times"][-1])
    ax.set_title("Burst Waveforms")
    if save_fig:
        f.savefig(f"{file_name}_Waveform_times.png", dpi=300, bbox_inches='tight')


    # --- Plot 6: Time-frequency map for a specific epoch with burst peaks overlay ---
    for trial in epochs_to_visualize:
        tr_map = bursts["trial"] == trial  # Get burst indices corresponding to selected trial

        # --- Declare numeric frequency array from config ---
        foi_numeric = np.linspace(config["min_freq"], config["max_freq"], config["n_freq"])
        
        tf_trials_fooof = tf_trials[trial][search_range] - ap[trial][search_range][:, np.newaxis]

        f, ax = plt.subplots(1, 1, figsize=(15, 7))
        spectrum = ax.imshow(
            tf_trials_fooof,#tf_trials[trial][search_range],
            cmap="turbo",
            origin="lower",
            extent=(times[0], times[-1], 
                foi_numeric[search_range[0]],       # lower frequency in band
                foi_numeric[search_range[-1]]),
            aspect="auto"#,
            #vmin=-0.00002, vmax=0.00006
        )

        # Overlay peak locations
        ax.scatter(bursts["peak_time"][tr_map], bursts["peak_freq"][tr_map], s=10, marker="+", c="black")

        # Add horizontal lines for frequency limits
        ax.axhline(freqs_lims[0], lw=0.5, c="black")
        ax.axhline(freqs_lims[-1], lw=0.5, c="black")

        # Label the axes
        ax.set_xlabel("Time (s)")
        #ax.set_xlim([1.2,2.2])
        ax.set_ylabel("Frequency (Hz)")

        # Add colorbar and layout adjustment
        plt.colorbar(spectrum, ax=ax)
        plt.tight_layout()


        if save_fig:
            f.savefig(f"{file_name}_epoch_{trial}_FOOOF_Peak.png", dpi=300, bbox_inches='tight')

    
    # --- Plot 7: Time-frequency map for a specific epoch with burst peaks overlay (no FOOOF) ---
    for trial in epochs_to_visualize:
        tr_map = bursts["trial"] == trial  # Get burst indices corresponding to selected trial

        f, ax = plt.subplots(1, 1, figsize=(15, 7))
        spectrum = ax.imshow(
            tf_trials[trial][search_range],
            cmap="turbo",
            origin="lower",
            extent=(times[0], times[-1], 
                foi_numeric[search_range[0]],       # lower frequency in band
                foi_numeric[search_range[-1]]),
            aspect="auto"
        )

        # Overlay peak locations
        ax.scatter(bursts["peak_time"][tr_map], bursts["peak_freq"][tr_map], s=10, marker="+", c="black")

        # Add horizontal lines for frequency limits
        ax.axhline(freqs_lims[0], lw=0.5, c="black")
        ax.axhline(freqs_lims[-1], lw=0.5, c="black")

        # Label the axes
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")

        # Add colorbar and layout adjustment
        plt.colorbar(spectrum, ax=ax)
        plt.tight_layout()

        if save_fig:
            f.savefig(f"{file_name}_epoch_{trial}_Peak.png", dpi=300, bbox_inches='tight')

    # --- Plot 8: Average time-frequency map for all epochs ---

    # --- Declare numeric frequency array from config ---
    
    # Subtract aperiodic from TF for all trials and average them
    tf_all_fooof = tf_trials[:, search_range, :] - ap[:, search_range][:, :, np.newaxis]
    mean_tf_all_fooof = tf_all_fooof.mean(axis=0)
    

    f, ax = plt.subplots(1, 1, figsize=(15, 7))
    spectrum = ax.imshow(
        mean_tf_all_fooof,
        cmap="turbo",
        origin="lower",
        extent=(times[0], times[-1], 
            foi_numeric[search_range[0]],       # lower frequency in band
            foi_numeric[search_range[-1]]),
        aspect="auto"
    )

    # Label the axes
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")

    # Add colorbar and layout adjustment
    plt.colorbar(spectrum, ax=ax)
    plt.tight_layout()


    if save_fig:
        f.savefig(f"{file_name}_epoch_average_trial.png", dpi=300, bbox_inches='tight')

    



## VII. Pipeline

In [ ]:
def pipeline(config, x, y, ch_names,file_path):
    """
    Runs the full processing pipeline on LFP data:
    - Loads signal and metadata
    - Segments into epochs
    - Computes Superlet transform and FOOOF fit
    - Extracts bursts
    - Visualizes results (PSD, bursts, waveforms, TFR)

    Parameters:
        config (dict): Configuration dictionary with all analysis parameters.
        x (list or array): X-coordinates of each channel.
        y (list or array): Y-coordinates of each channel.
        ch_names (list): List of channel names matching order in x/y.
        file_path : path of the neo .mat file to analyze

    Returns:
        None (results are saved to disk as plots and CSV/pickle files)
    """

    # --- Step 1: Load data and create MNE Raw/Epochs ---
    epoch_duration = config["epoch_duration"]

    print("##### Data Creation #####")
    raw, events, sfreq, _= create_neo_file(x, y, ch_names, epoch_duration,file_path)
    print("##### Data Created #####")

    # Create epochs from the raw signal
    epochs_raw = mne.Epochs(raw, events, event_id=1, tmin=0, tmax=epoch_duration - 1/sfreq, baseline=None)
    times = epochs_raw.times  # time vector per epoch

    # Find epochs that contain NaN and drop them
    has_nan = np.isnan(epochs_raw.get_data()).any(axis=(1, 2))
    epochs = epochs_raw.copy().drop(np.where(has_nan)[0])

    dropped_epoch_indices = np.where(has_nan)[0] # indices of dropped epochs
    kept_epoch_indices = np.where(~has_nan)[0] # indices of epochs that remain

    # Save dropped epoch indices for later use
    dropped_epochs_info = {
    "dropped_indices": dropped_epoch_indices.tolist(),
    "kept_indices": kept_epoch_indices.tolist(),
    "total_epochs_before": len(epochs_raw),
    "total_epochs_after": len(kept_epoch_indices)
    }

    # Save as a json file
    drop_save_path = os.path.join(f"{file_path[:-8]}_dropped_epochs.json")
    with open(drop_save_path, "w") as f:
        json.dump(dropped_epochs_info, f, indent=4)
    

    # --- Step 2: Select channel to analyse ---
    ch_to_analyse = config["ch_to_analyse"]
    index_ch = ch_names.index(ch_to_analyse)
    n_trials = (epochs.get_data()).shape[0]
    epoch_idx = np.arange(0, n_trials, config["epoch_step"]) #Define all analyzed epochs (every "epoch_step" epoch)
    trials = epochs.get_data()[epoch_idx, index_ch, :]  # Extract data for the selected channel only

    # --- Step 3: Superlet + FOOOF analysis ---
    done_superlet = config["Done_superlet"]
    max_freq = config["max_freq"]
    min_freq=config["min_freq"]
    freqs = np.linspace(min_freq, max_freq, 650)  # Frequencies of interest

    # Define frequency band to analyse
    Freqs_band = config["Freq_Bands"]
    freq_to_analyse = config["foi"]#config["fta"]
    freq_lims = Freqs_band[freq_to_analyse]

    # Define frequency window to search for bursts (±3 Hz margin)
    if freq_lims[0] - 3 < 1:
        upper_band = freq_lims[1] + 3
        search_range = np.where((freqs >= 1) & (freqs <= upper_band))[0]
    else:
        lower_band = freq_lims[0] - 3
        upper_band = freq_lims[1] + 3
        search_range = np.where((freqs >= lower_band) & (freqs <= upper_band))[0]


    print("##### Superlet Processing #####")
    tf_trials, freqs, ap, average_psd, offset, knee, exponent = superlet_analysis(config, done_superlet, file_path, freqs, trials, sfreq,ch_to_analyse)

    # --- Step 4: Burst detection ---
    burst = config["Do_burst_analysis"]
    done_burst = config["Done_burst"]
    save_csv = config["save_csv"]

    if burst:
        bursts = burst_extraction(
            done_burst, trials, tf_trials, search_range, times, freqs,
            freq_lims, ap, sfreq, file_path, ch_to_analyse, freq_to_analyse,
            save_csv=save_csv
        )

    # --- Step 5: Visualization ---
    plot_choice = config["visualisation_for_a_specific_band"]
    save_fig = config["save_fig"]

    if not plot_choice:
        config["foi"]=""
        # Default visualization in the current analysis band
        plot_psd(freqs, average_psd, ap, file_path, config, search_range)
        plot_bursts(bursts, file_path, config, tf_trials, search_range, times, freqs, freq_lims, ap, save_fig)
    else:
        # Alternative visualization in a different band specified by "foi"
        foi = config["foi"]
        freq_lims = Freqs_band[foi]
        bursts = filtered_burst_to_foi(bursts, freq_lims[0], freq_lims[-1])

        if freq_lims[0] - 3 < 1:
            upper_band = freq_lims[1] + 3
            search_range = np.where((freqs >= 1) & (freqs <= upper_band))[0]
        else:
            lower_band = freq_lims[0] - 3
            upper_band = freq_lims[1] + 3
            search_range = np.where((freqs >= lower_band) & (freqs <= upper_band))[0]

        plot_psd(freqs, average_psd, ap, file_path, config, search_range)
        plot_bursts(bursts, file_path, config, tf_trials, search_range, times, freqs, freq_lims, ap, save_fig)


## VIII. Run

### 1) Config

In [ ]:
config = {
    "epoch_duration": 9.099,  # Duration of each epoch in seconds

    "Done_superlet": True,  # If True, load existing Superlet results (.npz); else compute and save
    "Done_burst": False,     # If True, load existing burst data (.pkl); else compute and save

    "max_freq": 100,         # Maximum frequency for time-frequency decomposition
    "min_freq" : 1,          # Minimum frequency for time-frequency decomposition
    "n_freq" : 650,      # Number of frequency points

    "Visualization": True,    # Typo: should be "Visualization" or remove (not used in pipeline)

    "Freq_Bands": {          # Frequency bands used for filtering and plotting
        "delta": [1.0, 4.0],
        "theta": [4.0, 8.0],
        "alpha": [8.0, 13.0],
        "beta": [13.0, 30.0],
        "low_gamma": [30.0, 50.0],
        "high_gamma": [50.0, 80.0],
        "gamma": [30.0, 80.0],
        "personalized": [1, 100.0]
    },

    "fta": "personalized",   # Frequency band to analyze (for extraction and Superlet)
    "foi": "low_gamma",          # Frequency band to visualize (for plotting PSD and bursts)

    "ch_to_analyse": "CH28",  # Name of the channel to analyze
    "epochs_to_visualize": [20,60],  # Index of the epoch to visualize in the final TF map
    "epoch_step" : 1, #Number of epochs between analysed epoch (analyzing every epoch_step epoch)

    "Do_burst_analysis": True,     # If True, perform burst detection
    "visualisation_for_a_specific_band": True,  # If True, plots are done for `foi`; else for `fta`

    "save_fig": True,  # Whether to save the generated figures
    "save_csv": True   # Whether to export burst features as CSV
}

### 2) Run

In [ ]:
all_file_paths = file_choice()

In [ ]:
for file_path in all_file_paths : 
    pipeline(config,x,y,ch_names,file_path)